# NLP – 4 (BERT Fine-Tuning)
Dataset = IMDB Movie Reviews from Kaggle

In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn matplotlib seaborn

In [ ]:
import kagglehub
import pandas as pd

# 1. Download the 50k Reviews dataset (Standard for BERT training)
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

# 2. Load the CSV (kagglehub returns the folder path, so we append the filename)
# Usually, the file is named 'IMDB Dataset.csv'
df = pd.read_csv(f"{path}/IMDB Dataset.csv")

# 3. Quick Preprocessing for BERT
# Map 'positive' to 1 and 'negative' to 0
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df = df.rename(columns={'review': 'text'})

print(df.head())

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate

# 1. Load the ur Kaggle Dataset

# 2. Split Data (80/10/10)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42)

# Convert to Hugging Face Dataset format
ds = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df)
})

In [ ]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = ds.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load Model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

In [ ]:
def prepare_experiment(model, mode="full"):
    if mode == "frozen":
        # Freeze ALL BERT layers
        for param in model.bert.parameters():
            param.requires_grad = False
    elif mode == "last_two":
        # Freeze all except last 2 layers
        for param in model.bert.encoder.layer[:-2].parameters():
            param.requires_grad = False
    return model

# Example: Run the "last two layers" experiment
model = prepare_experiment(model, mode="last_two")

In [ ]:
# 1. Define the metrics
metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 2. Set up training arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",      # Ensure this is 'eval_strategy'
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# 3. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,      # The tokenizer is already inside here
    compute_metrics=compute_metrics,
)

# 4. Start training
trainer.train()

In [ ]:
# Get predictions on the Test set
predictions = trainer.predict(tokenized_datasets["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Print Final Metrics
print(f"Final Test Metrics: {predictions.metrics}")

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Neg", "Pos"])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix: BERT Fine-Tuning")
plt.show()